# 02 — LSTM

Notebook ini berdiri sendiri: jalankan `make preprocess` sekali dulu
(datanya diambil dari BigQuery), lalu Run All di sini untuk train + evaluasi
+ upload model LSTM saja.

Input-nya tabel `preprocessed_data_hourly` + `preprocessing_metadata_hourly`
hasil `make preprocess`: semua stasiun + kalender (sin/cos) untuk semua model,
ditambah akumulasi hujan DAS milik stasiun target. `USE_RAINFALL = False`
menjalankan ablation tanpa hujan; hasilnya disimpan sebagai `<model>_norain`.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import io
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

# False = ablation tanpa fitur hujan; hasil disimpan sebagai <model>_norain.
USE_RAINFALL = True

PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_hourly"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_hourly"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# --- Metadata: kolom kalender/stasiun/fitur + mapping nama kolom BigQuery ---
meta_row = client.query(f"""
    SELECT payload
    FROM `{METADATA_TABLE}`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])
station_cols  = metadata["station_cols"]
column_map    = metadata["column_map"]                 # station -> kolom BigQuery
reverse_column_map = {v: k for k, v in column_map.items()}

# Fitur: semua stasiun + kalender (sin/cos) untuk semua model, ditambah
# akumulasi hujan DAS milik stasiun target (lihat `make preprocess`).
if "base_feature_cols" not in metadata:
    raise RuntimeError("Metadata versi lama (tanpa fitur hujan/kalender sin-cos); jalankan `make preprocess` dulu.")
BASE_FEATURE_COLS    = metadata["base_feature_cols"]
RAIN_COLS_BY_STATION = metadata["rain_cols_by_station"] if USE_RAINFALL else {}
FEATURES_BY_STATION  = {s: BASE_FEATURE_COLS + RAIN_COLS_BY_STATION.get(s, []) for s in station_cols}
ALL_FEATURE_COLS     = BASE_FEATURE_COLS + [c for cols in RAIN_COLS_BY_STATION.values() for c in cols]


def features_for(target):
    """Kolom input model untuk stasiun target (urutan sama dengan serving)."""
    return FEATURES_BY_STATION[target]


# --- df_train / df_val / df_test dari tabel preprocessed_data --------------
def load_split(split):
    query = f"""
        SELECT *
        FROM `{PREPROCESSED_TABLE}`
        WHERE split = @split
        ORDER BY datetime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("split", "STRING", split),
    ])
    df = client.query(query, job_config=job_config).to_dataframe()
    df = df.set_index("datetime").rename(columns=reverse_column_map)
    return df[ALL_FEATURE_COLS]


df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")

print(f"Stations : {station_cols}")
n_rain = {s: len(c) for s, c in RAIN_COLS_BY_STATION.items()}
print(f"Features : {len(BASE_FEATURE_COLS)} dasar + fitur hujan per stasiun: {n_rain} (USE_RAINFALL={USE_RAINFALL})")
print(f"Train/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,} rows")

## Konfigurasi BigQuery — Hasil Model

Tabel hasil untuk model ini (semua di dataset `tables`):

- `lstm_val_metrics_hourly` — metrik validasi per (station, horizon, metric).
- `lstm_weights_hourly` — bobot terlatih per (stasiun, horizon), disimpan sebagai `BYTES`.
- `lstm_predictions_hourly` — prediksi test-set per (station, horizon, datetime).
- `lstm_metrics_hourly` — metrik test-set per (station, horizon, metric).

Bobot ditulis ke BigQuery segera setelah training (bukan ke disk), sehingga
langkah test-set evaluation di bawah bisa memuat ulang bobot dari BigQuery
saja bila objek model tidak lagi ada di memori (mis. notebook dibuka ulang).

In [ ]:
MODEL_NAME = "lstm" if USE_RAINFALL else "lstm_norain"

VAL_METRICS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_val_metrics_hourly"
WEIGHTS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_weights_hourly"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_predictions_hourly"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_metrics_hourly"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Val metrics ->", VAL_METRICS_TABLE)
print("Weights     ->", WEIGHTS_TABLE)
print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOOKBACK   = 72      # jam
HORIZONS   = [1, 3, 12]  # jam ke depan
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 7       # early stopping

print(f"Device     : {DEVICE}")
print(f"Lookback   : {LOOKBACK} jam")
print(f"Models     : {len(station_cols)} stasiun \u00d7 {len(HORIZONS)} horizon = {len(station_cols)*len(HORIZONS)}")

In [ ]:
class TMADataset(Dataset):
    def __init__(self, df, target_station, feature_cols, lookback, horizon):
        self.X, self.y = [], []
        data   = df[feature_cols].values.astype(np.float32)
        target = df[target_station].values.astype(np.float32)

        for i in range(lookback, len(data) - horizon + 1):
            self.X.append(data[i - lookback:i])          # (lookback, n_features)
            self.y.append(target[i + horizon - 1])        # scalar t+h

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)   # ambil timestep terakhir

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            total_loss += criterion(pred, y).item() * len(y)
    return total_loss / len(loader.dataset)


def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).cpu().numpy())
            trues.append(y.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE \u2014 symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE \u2014 Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
lstm_models = {}   # {station: {horizon: model}}
lstm_results = {}   # {station: {horizon: metrics}}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    lstm_models[target] = {}
    lstm_results[target] = {}

    for h in HORIZONS:
        train_ds = TMADataset(df_train, target, features_for(target), LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, features_for(target), LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = LSTMForecaster(n_features=len(features_for(target))).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}h | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        model.load_state_dict(best_state)
        lstm_models[target][h] = model

        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        lstm_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h \u2192 RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")

In [ ]:
weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
val_metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]


def serialize_torch_state(state_dict):
    """Serialize state_dict PyTorch ke bytes (in-memory, tanpa file sementara)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getvalue()


saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []
val_metric_rows = []
for station, horizons in lstm_models.items():
    for h, model in horizons.items():
        blob = serialize_torch_state(model.state_dict())
        weight_rows.append({
            "model": MODEL_NAME, "station": station, "horizon": int(h),
            "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
        })
    for hkey, metrics in lstm_results[station].items():
        h = int(hkey.replace("h+", ""))
        for metric_name, value in metrics.items():
            val_metric_rows.append({
                "model": MODEL_NAME, "station": station, "horizon": h,
                "metric": metric_name, "value": float(value),
            })

weights_df = pd.DataFrame(weight_rows)
val_metrics_df = pd.DataFrame(val_metric_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total).")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
load_df(val_metrics_df, VAL_METRICS_TABLE, val_metric_schema)
print(f"\n\u2714 Bobot + metrik validasi '{MODEL_NAME}' tersimpan di BigQuery.")

## Test Set Evaluation

In [ ]:
def _load_weights_from_bq(station, h):
    query = f"""
        SELECT weights
        FROM `{WEIGHTS_TABLE}`
        WHERE station = @station AND horizon = @horizon
        ORDER BY saved_at DESC
        LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("station", "STRING", station),
        bigquery.ScalarQueryParameter("horizon", "INT64", h),
    ])
    row = client.query(query, job_config=job_config).to_dataframe()
    if row.empty:
        raise RuntimeError(f"Bobot utk ({station}, h+{h}) tidak ditemukan di {WEIGHTS_TABLE}.")
    return row.iloc[0]["weights"]


def predict_test_lstm(target, h):
    test_ds = TMADataset(df_test, target, features_for(target), LOOKBACK, h)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    if target in lstm_models and h in lstm_models[target]:
        model = lstm_models[target][h]
    else:
        # rebuild + load bobot dari BigQuery (mis. notebook baru dibuka ulang)
        model = LSTMForecaster(n_features=len(features_for(target))).to(DEVICE)
        blob = _load_weights_from_bq(target, h)
        model.load_state_dict(torch.load(io.BytesIO(blob), map_location=DEVICE))

    idx = df_test.index[LOOKBACK + h - 1 : len(df_test)]
    y_pred, y_true = get_predictions(model, test_loader)
    return idx[:len(y_pred)], y_true, y_pred


test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        idx, y_true, y_pred = predict_test_lstm(target, h)
        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": MODEL_NAME,
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows.")
test_pred_df.head()

In [ ]:
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": MODEL_NAME, "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)
print(f"Test metrics rows: {len(test_metrics_df)}")

## Write Results to BigQuery

Prediksi dan metrik test-set model ini ditulis ke BigQuery. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials,
mis. kredensial bawaan Colab). Mode tulis default `WRITE_TRUNCATE` (ganti
seluruh isi tabel); ubah `WRITE_DISPOSITION` di atas ke `WRITE_APPEND` bila
ingin menumpuk hasil beberapa run.

In [ ]:
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

print("Uploading to BigQuery ...")
load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n\u2714 BigQuery write selesai.")

## Interpretasi Model (XAI)

LSTM adalah *black box*, jadi alasan di balik prediksinya dijelaskan dengan dua
metode yang saling melengkapi (keduanya dihitung pada **test set**, dalam skala
terpreprocess 0–1):

1. **Permutation Feature Importance (PFI)** — satu fitur (seluruh window 72 jam-nya)
   diacak antar-sampel, lalu diukur seberapa besar RMSE naik. Model-agnostic dan
   langsung mengukur ketergantungan model pada fitur itu. Pasangan `*_sin`/`*_cos`
   diacak bersamaan sebagai satu fitur kalender.
2. **Integrated Gradients (IG)** — atribusi berbasis gradien per (jam, fitur) relatif
   terhadap baseline = rata-rata fitur di train set. Atribusi bersifat aditif:
   jumlahnya = `f(x) − f(baseline)` (kolom `ig_rel_gap` memeriksa ini; mendekati 0
   berarti aproksimasi integral cukup akurat, naikkan `XAI_IG_STEPS` bila besar).
   Dari IG diturunkan (a) kontribusi bersih per fitur, (b) profil temporal — seberapa
   jauh ke belakang model melihat, dan (c) penjelasan lokal untuk event TMA puncak.

**Cara membaca & batasan**
- Importance menunjukkan apa yang *dipakai model*, bukan hubungan sebab-akibat di DAS.
  Fitur yang berkorelasi (mis. TMA stasiun hulu & hilir, atau akumulasi hujan 3 jam vs 6 jam)
  dapat saling menggantikan sehingga importance-nya terbagi.
- Bila PFI dan IG sepakat (Spearman tinggi) kesimpulannya lebih kokoh; bila berbeda,
  periksa dulu sebelum menafsirkan.
- Fitur hujan hanya ada bila `USE_RAINFALL = True`.
- Bagian ini ditaruh setelah penulisan ke BigQuery sehingga kegagalan XAI tidak
  memengaruhi hasil yang sudah tersimpan. Waktu hitung dapat dikurangi lewat
  `XAI_STATIONS`, `XAI_MAX_WINDOWS`, dan `XAI_IG_WINDOWS`.


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# --- Konfigurasi XAI -----------------------------------------------------------
XAI_STATIONS      = station_cols     # isi subset stasiun untuk mempercepat
XAI_MAX_WINDOWS   = 1500             # window test untuk permutation importance
XAI_PFI_REPEATS   = 3
XAI_IG_WINDOWS    = 200              # window test untuk Integrated Gradients
XAI_IG_STEPS      = 32
XAI_TOP_K         = 10
XAI_LOCAL_HORIZON = HORIZONS[1]      # horizon utk penjelasan lokal (event puncak)
XAI_SEED          = 42

GROUP_COLORS = {"TMA stasiun": "#1f77b4", "Hujan DAS": "#2ca02c", "Kalender": "#9e9e9e"}


def feature_group(name):
    if name in station_cols:
        return "TMA stasiun"
    if name.startswith("rain_"):
        return "Hujan DAS"
    return "Kalender"


def feature_units(feats):
    """Unit interpretasi: pasangan <x>_sin/<x>_cos dianggap satu fitur
    (diacak dan dijumlahkan bersama) supaya encoding siklik tidak terpecah."""
    units = {}
    for i, name in enumerate(feats):
        key = name[:-4] if name.endswith(("_sin", "_cos")) else name
        units.setdefault(key, []).append(i)
    return units


def load_trained_model(target, h):
    """Model terlatih dari memori, atau dari BigQuery bila notebook dibuka ulang."""
    if target in lstm_models and h in lstm_models[target]:
        model = lstm_models[target][h]
    else:
        model = LSTMForecaster(n_features=len(features_for(target))).to(DEVICE)
        blob = _load_weights_from_bq(target, h)
        model.load_state_dict(torch.load(io.BytesIO(blob), map_location=DEVICE))
    return model.eval()


def even_subset(n, k):
    """k indeks merata di sepanjang n window (menjaga sebaran waktu test set)."""
    return np.unique(np.linspace(0, n - 1, min(n, k)).astype(int))


@torch.no_grad()
def predict_tensor(model, X, batch=512):
    return torch.cat([model(X[i:i + batch].to(DEVICE)).cpu() for i in range(0, len(X), batch)])


def permutation_importance(model, X, y, units, n_repeats, seed):
    """Kenaikan RMSE saat satu unit fitur diacak antar-window (seluruh 72 jam
    fitur itu ditukar dengan milik window lain). Makin besar = makin penting;
    ~0 atau negatif = model tidak bergantung pada fitur tsb."""
    rng = np.random.default_rng(seed)
    rmse = lambda Xt: torch.sqrt(torch.mean((predict_tensor(model, Xt) - y) ** 2)).item()
    base = rmse(X)
    imp = np.zeros((len(units), n_repeats))
    for u, idx in enumerate(units.values()):
        for r in range(n_repeats):
            Xp = X.clone()
            perm = torch.from_numpy(rng.permutation(len(X)))
            Xp[:, :, idx] = X[perm][:, :, idx]
            imp[u, r] = rmse(Xp) - base
    return base, imp


def integrated_gradients(model, X, baseline, steps, batch=32):
    """Integrated Gradients (aturan titik-tengah) terhadap output model.
    Mengembalikan atribusi (N, L, F) dan selisih kelengkapan per window:
    sum(atribusi) seharusnya = f(x) - f(baseline)."""
    attrs, gaps = [], []
    base_row = baseline.view(1, 1, -1).to(DEVICE)
    alphas = (torch.arange(steps, dtype=torch.float32) + 0.5) / steps
    # cuDNN RNN hanya mengizinkan backward di mode train -> matikan cuDNN,
    # model tetap eval() sehingga dropout tidak aktif.
    with torch.backends.cudnn.flags(enabled=False):
        for i in range(0, len(X), batch):
            xb = X[i:i + batch].to(DEVICE)
            bb = base_row.expand_as(xb).contiguous()
            grad_sum = torch.zeros_like(xb)
            for a in alphas:
                xi = (bb + a.item() * (xb - bb)).requires_grad_(True)
                (g,) = torch.autograd.grad(model(xi).sum(), xi)
                grad_sum += g
            attr = (xb - bb) * grad_sum / steps
            with torch.no_grad():
                gap = model(xb) - model(bb) - attr.sum(dim=(1, 2))
            attrs.append(attr.detach().cpu())
            gaps.append(gap.cpu())
    return torch.cat(attrs).numpy(), torch.cat(gaps).numpy()


# --- Hitung PFI + IG untuk setiap (stasiun, horizon) ---------------------------
pfi_rows, ig_rows = [], []
ig_time, ig_local = {}, {}

for target in XAI_STATIONS:
    feats = features_for(target)
    units = feature_units(feats)
    unit_names = list(units)
    # Baseline IG = rata-rata tiap fitur di train set (input "tipikal", skala preprocessing)
    baseline = torch.tensor(df_train[feats].to_numpy(np.float32).mean(axis=0))

    for h in HORIZONS:
        model = load_trained_model(target, h)
        ds = TMADataset(df_test, target, feats, LOOKBACK, h)
        X_all, y_all = ds.X, ds.y

        # (1) Permutation feature importance
        sel = even_subset(len(X_all), XAI_MAX_WINDOWS)
        base_rmse, imp = permutation_importance(model, X_all[sel], y_all[sel], units,
                                                XAI_PFI_REPEATS, XAI_SEED)
        for u, name in enumerate(unit_names):
            pfi_rows.append({"station": target, "horizon": h, "feature": name,
                             "pfi_mean": imp[u].mean(), "pfi_std": imp[u].std(),
                             "base_rmse": base_rmse})

        # (2) Integrated Gradients
        sel_ig = even_subset(len(X_all), XAI_IG_WINDOWS)
        attr, gap = integrated_gradients(model, X_all[sel_ig], baseline, XAI_IG_STEPS)
        unit_attr = np.stack([attr[:, :, idx].sum(-1) for idx in units.values()], axis=-1)  # (N, L, U)
        ig_net = np.abs(unit_attr.sum(axis=1)).mean(axis=0)            # (U,) |kontribusi bersih| rata-rata
        ig_time[(target, h)] = np.abs(unit_attr).sum(axis=2).mean(axis=0)   # (L,)
        delta_f = np.abs(attr.sum(axis=(1, 2)) + gap).mean()           # |f(x) - f(baseline)|
        rel_gap = float(np.abs(gap).mean() / (delta_f + 1e-12))
        for u, name in enumerate(unit_names):
            ig_rows.append({"station": target, "horizon": h, "feature": name,
                            "ig_net": float(ig_net[u]), "ig_rel_gap": rel_gap})

        rho = spearmanr(imp.mean(axis=1), ig_net).correlation
        print(f"{target:<20} h+{h:<2} | RMSE={base_rmse:.4f} | IG rel.gap={rel_gap:.3%} | "
              f"Spearman(PFI, IG)={rho:.2f}")

        # (3) Penjelasan lokal: window dengan TMA tertinggi di test set
        if h == XAI_LOCAL_HORIZON:
            i_peak = int(torch.argmax(y_all))
            a_peak, _ = integrated_gradients(model, X_all[i_peak:i_peak + 1], baseline, XAI_IG_STEPS)
            ig_local[target] = {
                "units": unit_names,
                "unit_attr": np.stack([a_peak[0][:, idx].sum(-1) for idx in units.values()], axis=-1),  # (L, U)
                "time": df_test.index[LOOKBACK + h - 1 + i_peak],
                "y_true": float(y_all[i_peak]),
                "y_pred": float(predict_tensor(model, X_all[i_peak:i_peak + 1])[0]),
            }

xai_df = pd.DataFrame(pfi_rows).merge(pd.DataFrame(ig_rows), on=["station", "horizon", "feature"])
xai_df.insert(0, "model", MODEL_NAME)
xai_df["group"] = xai_df["feature"].map(feature_group)
# kesepakatan peringkat PFI vs IG per (stasiun, horizon)
xai_df["spearman_pfi_ig"] = xai_df.groupby(["station", "horizon"])["pfi_mean"].transform(
    lambda s: spearmanr(s, xai_df.loc[s.index, "ig_net"]).correlation)
print(f"\nxai_df: {len(xai_df)} baris (model, station, horizon, feature)")
xai_df.head()


In [ ]:
# --- Plot global ---------------------------------------------------------------
def barh_top_k(ax, d, col, err=None):
    d = d.nlargest(XAI_TOP_K, col).iloc[::-1]
    ax.barh(d["feature"], d[col], xerr=None if err is None else d[err],
            color=[GROUP_COLORS[g] for g in d["group"]], error_kw={"lw": 0.8})
    ax.tick_params(axis="y", labelsize=8)


legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]

# 1) Per stasiun: top fitur menurut PFI (baris atas) dan IG (baris bawah), per horizon
for target in XAI_STATIONS:
    fig, axes = plt.subplots(2, len(HORIZONS), figsize=(5.2 * len(HORIZONS), 7), squeeze=False)
    for j, h in enumerate(HORIZONS):
        d = xai_df[(xai_df.station == target) & (xai_df.horizon == h)]
        barh_top_k(axes[0, j], d, "pfi_mean", "pfi_std")
        axes[0, j].set_title(f"h+{h}  |  PFI (kenaikan RMSE)", fontsize=10)
        barh_top_k(axes[1, j], d, "ig_net")
        axes[1, j].set_title(f"h+{h}  |  IG (|kontribusi bersih|)", fontsize=10)
    fig.legend(legend_handles, list(GROUP_COLORS), loc="lower center", ncol=len(GROUP_COLORS), frameon=False)
    fig.suptitle(f"{MODEL_NAME.upper()} — fitur terpenting: {target}")
    fig.tight_layout(rect=[0, 0.04, 1, 0.97])
    plt.show()

# 2) Seberapa jauh ke belakang model "melihat"? (profil atribusi IG per jam-lag)
n_cols = 3
n_rows = int(np.ceil(len(XAI_STATIONS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 3.4 * n_rows), squeeze=False, sharex=True)
for ax in axes.ravel():
    ax.axis("off")
lags = np.arange(-LOOKBACK, 0)
for ax, target in zip(axes.ravel(), XAI_STATIONS):
    ax.axis("on")
    for h in HORIZONS:
        ax.plot(lags, ig_time[(target, h)], label=f"h+{h}")
    ax.set_title(target, fontsize=10)
    ax.set_xlabel("jam sebelum waktu prediksi")
    ax.set_ylabel("mean sum|atribusi|")
axes.ravel()[0].legend(frameon=False)
fig.suptitle(f"{MODEL_NAME.upper()} — profil temporal atribusi (Integrated Gradients)")
fig.tight_layout()
plt.show()

# 3) Porsi kelompok fitur (dari IG), per stasiun dan horizon
share = (xai_df.assign(share=xai_df["ig_net"] / xai_df.groupby(["station", "horizon"])["ig_net"].transform("sum"))
               .groupby(["station", "horizon", "group"])["share"].sum().unstack("group").fillna(0))
share = share.reindex(columns=[g for g in GROUP_COLORS if g in share.columns])
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 0.55 * len(XAI_STATIONS) + 1.8),
                         squeeze=False, sharey=True)
for ax, h in zip(axes[0], HORIZONS):
    share.xs(h, level="horizon").loc[XAI_STATIONS].plot.barh(
        stacked=True, ax=ax, color=[GROUP_COLORS[g] for g in share.columns], legend=False)
    ax.set_title(f"h+{h}")
    ax.set_xlim(0, 1)
    ax.invert_yaxis()
    ax.set_ylabel("")
fig.legend(legend_handles[:len(share.columns)], list(share.columns), loc="lower center",
           ncol=len(share.columns), frameon=False)
fig.suptitle(f"{MODEL_NAME.upper()} — porsi atribusi per kelompok fitur")
fig.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

# 4) Ringkasan: 3 fitur teratas per (stasiun, horizon) + kesepakatan PFI vs IG
summary = (xai_df.sort_values("pfi_mean", ascending=False)
                 .groupby(["station", "horizon"])
                 .agg(top3_pfi=("feature", lambda s: ", ".join(s.head(3))),
                      spearman_pfi_ig=("spearman_pfi_ig", "first"),
                      ig_rel_gap=("ig_rel_gap", "first")))
summary


In [ ]:
# --- Penjelasan lokal: window dengan TMA tertinggi di test set -----------------
# Heatmap atribusi IG (jam ke-belakang x fitur). Merah = mendorong prediksi naik
# relatif terhadap baseline, biru = menurunkan. Sumbu x: -72 = terjauh, -1 = jam terakhir.
n_cols = 2
n_rows = int(np.ceil(len(XAI_STATIONS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7.5 * n_cols, 3.6 * n_rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")

for ax, target in zip(axes.ravel(), XAI_STATIONS):
    loc = ig_local[target]
    A = loc["unit_attr"]                                   # (L, U)
    top = np.argsort(-np.abs(A.sum(axis=0)))[:8]           # 8 fitur dgn kontribusi bersih terbesar
    vmax = np.abs(A[:, top]).max() + 1e-12
    ax.axis("on")
    im = ax.imshow(A[:, top].T, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   extent=[-LOOKBACK, 0, len(top), 0])
    ax.set_yticks(np.arange(len(top)) + 0.5)
    ax.set_yticklabels([loc["units"][i] for i in top], fontsize=8)
    ax.set_xlabel("jam sebelum waktu prediksi")
    ax.set_title(f"{target}  |  {loc['time']:%Y-%m-%d %H:%M} (h+{XAI_LOCAL_HORIZON})\n"
                 f"aktual={loc['y_true']:.3f}  prediksi={loc['y_pred']:.3f}", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

fig.suptitle(f"{MODEL_NAME.upper()} — penjelasan lokal (Integrated Gradients) untuk TMA puncak di test set", y=1.0)
fig.tight_layout()
plt.show()
